In [2]:
words= open("data/shakespeare_100k.txt", "r").read().splitlines()

In [ ]:
import torch

N = torch.zeros((52,52),dtype=torch.int32) 

lookup=sorted(list(set(''.join(words))))
stoi={s:i+1 for i,s in enumerate(lookup)}
stoi['.']=0
itos={i:s for s,i in stoi.items()}

# initialise N matrix

for w in words:
    chs=["."] + list(w) + ["."] 
    for ch1, ch2 in zip(chs,chs[1:]):
        ix1=stoi[ch1]
        ix2=stoi[ch2]
        N[ix1,ix2]+=1


P=(N+1).float() # did (N+1) it does scalar addition of 1 to every elemnt of co-occurrence matrix so no prabability is exactly zero it is called model smoothing
P=P/P.sum(1,keepdim=True) # converting numbers in the matrix to probabilities

g= torch.Generator().manual_seed(214748367)

for i in range(20):
    out=[]
    ix=0
    while True:
        p=P[ix].float() # P is the matrix containing the probability distribution
        ix=torch.multinomial(p,num_samples=1,replacement=True,generator=g).item()

        out.append(itos[ix])

        if ix==0:
            break
    print(''.join(out))



# Loss function bigram model 

In [ ]:
log_likelihood=0.0
n=0.0
for w in words:
    chs=["."] + list(w) + ["."] 
    for ch1, ch2 in zip(chs,chs[1:]):
        ix1=stoi[ch1]
        ix2=stoi[ch2]
        prob=P[ix1, ix2]
        logprob=torch.log(prob) # log probabilities
        log_likelihood += logprob
        n+=1
        print(f"{ch1}{ch2}:  {prob:.4f}  {logprob:.4f}")

print(f'{log_likelihood=}') # our goal will be to minimize the negative log_likelihood
nll=-log_likelihood
avg_nll=(f"{nll/n}")# this is the avg negative log likelihood and we can minimize this instead
print(f'{avg_nll=}')

.t:  0.1164  -2.1510
th:  0.3312  -1.1049
he:  0.3511  -1.0467
ey:  0.0132  -4.3248
y.:  0.6657  -0.4069
log_likelihood=tensor(-9.0343)
avg_nll='1.8068593740463257'


Till now we used co-occurence matrix to predict the next char but we are going to implement the bigram in Neural Network settings with paramaters so that we can minize the loss and make better model for predicting next char 

In [ ]:
# creating the training set of bigram(x,y)
# this will create a dataset with every bigram ex for a bigram 'wi' x=w and y=i similarly for all possible pair
xs, ys = [], []

for w in words[:1]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        print(ch1,ch2)
        xs.append(ix1)
        ys.append(ix2)
xs = torch.tensor(xs)
ys = torch.tensor(ys)


. T
T H
H E
E .


In [20]:
xs

tensor([ 0, 20,  8,  5])

In [21]:
ys

tensor([20,  8,  5,  0])

In [24]:
# now our xs contains numerical (itegers) value mapped to a numerical value in ys but integer itself 
# can not be fed into the neural network we nee some encoding to represent that numerical value with
# context of it being a word in our vocabulary of 51 words we can different encodings such as one-hot,
# learned embeddings,etc

#one-hot embedding

import torch.nn.functional as F 
xenc=F.one_hot(xs, num_classes=51)

xenc.shape


torch.Size([4, 51])

In [34]:
# this is learned embedding in this embedding each character is represented by 8dim vector
# for example word s = [0.23,1.3,0.97,2.4......] unlike one hot encoding where evry word is represented using 51dim one-hot vector
# this embedding is better so we would use this
import torch
import torch.nn as nn

# Vocabulary size and embedding dimensions
vocab_size = 51
embedding_dim = 8

# Create embedding layer
embedding = nn.Embedding(vocab_size, embedding_dim)

# Convert input indices into embeddings
X = embedding(xs)

print(X)

tensor([[ 1.3369,  0.9466,  1.0702, -1.5245, -2.3324, -0.3221, -1.5168, -1.3828],
        [-1.6562, -0.0928, -0.9599, -1.0482, -0.3991,  1.0544, -0.1454, -0.1553],
        [-1.0358,  0.9162,  0.7969,  1.4693,  0.3994,  0.4067,  0.8983,  2.0599],
        [ 1.4752, -0.6599, -1.5528, -0.9655,  0.5380, -0.7259,  0.0504, -1.8598]],
       grad_fn=<EmbeddingBackward0>)


In [ ]:
# initialising the weights

W=torch.randn((8,20)) # this creates 8X20 matrix of weights which will be multiplied by our embedding vector here 20 represents 20 neurons in the input layer
                     # therefore size 8X20 as 51X8 * 8X20 = 51X20 output 
                     # torch.randn gives random numbers from a normal distribution with mean=0


what W contains is 8 rows with 20 columns that there are 8 different weight vectors containing 20 different weights for each neuron 

[
    [w1-1,w1-2,w1-3......w1-20],
    [w2-1,w2-2,w2-3......w2-20],
    .
    .
    .
    ......................w8-20]
]

where w2-3 = w2 of neuron 3
      w4-13 = w4 of neuron 13

In [37]:
(X@W)[1]

tensor([ 2.5502,  6.0701,  1.6813, -3.3388,  0.9443, -2.0230,  4.1858, -1.2046,
         3.4702, -0.2756,  2.3613,  4.9016,  1.8241, -7.3937,  1.5557,  4.1117,
        -4.9599, -1.9490,  0.2945, -3.0966], grad_fn=<SelectBackward0>)

In [ ]:
(X@W)[2,13]# this gives fiering rate of 13th neuron considering the 2nd input
# Basically it is the multiplication(dot prod) of 3rd row of the embedding vector and weights vector for the 13th neuron
# @ is used to compute matrix multiplication

tensor(2.8852, grad_fn=<SelectBackward0>)